# Fine-tuning Faster R-CNN model for Hazard Placard Detection

This notebook demonstrates how to fine-tune a Faster R-CNN model with a ResNet-101-FPN backbone using our packaged training module.

In [ ]:
import sys
import os
from pathlib import Path

# Add src directory to path
sys.path.append(str(Path().resolve().parent / "src"))

In [ ]:
import torch
from torch.utils.data import DataLoader
from pycocotools.coco import COCO

from un_detector.data.datasets import HazmatDataset
from un_detector.data.augmentation import get_augmented_transform
from un_detector.models.faster_rcnn import get_faster_rcnn_model
from un_detector.training.trainers import FasterRCNNTrainer
from un_detector.training.train_utils import collate_fn, create_subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Create Datasets & Dataloaders

In [ ]:
# Initialize datasets
train_dataset = HazmatDataset(
    data_dir="../data/data_faster_rcnn/train",
    annotations_file="../data/data_faster_rcnn/train/annotations/instances_train.json",
    transforms=get_augmented_transform(train=True)
)

val_dataset = HazmatDataset(
    data_dir="../data/data_faster_rcnn/val",
    annotations_file="../data/data_faster_rcnn/val/annotations/instances_val.json",
    transforms=get_augmented_transform(train=False)
)

# Optional: use subset if debugging (e.g. percentage = 0.1)
train_subset = create_subset(train_dataset, percentage=1.0)
val_subset = create_subset(val_dataset, percentage=1.0)

train_loader = DataLoader(
    train_subset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

# Load ground truth validation annotations for metrics computation
coco_val_gt = COCO("../data/data_faster_rcnn/val/annotations/instances_val.json")

## Initialize Model, Optimizer, and Trainer

In [ ]:
# ResNet-101 Backbone model initialization
model = get_faster_rcnn_model(num_classes=2, backbone_name="resnet101", pretrained_backbone=True)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
scaler = torch.cuda.amp.GradScaler()

trainer = FasterRCNNTrainer(
    model=model,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    device=device,
    scaler=scaler,
    checkpoint_dir="../data/models"
)

## Run Training

In [ ]:
num_epochs = 10
results = trainer.fit(
    epochs=num_epochs,
    train_loader=train_loader,
    val_loader=val_loader,
    coco_val_gt=coco_val_gt
)